# JOILang Cloud-only-by-config Advisor Compression Analysis — A6000/A100

This notebook is configured to run **cloud-only-by-config** on the current JOILang codebase.

It does **not** require strict cloud-only CLI flags such as `--cloud-only-advisor-compression`.
Instead, it emulates cloud-only advisor compression by configuration:

- `use_advisor=True`
- `advisor_compression_child_quota > 0`
- local compression lanes disabled by quota/ratio:
  - `compression_child_quota=0`
  - `compression_child_ratio=0.0`
  - `micro/block/multi/global compression quotas = 0`
- final gate verifies that no `source="compression_fallback"` rows were created.

This is the correct mode when the repository has the existing staged strong compression flags but does not yet have the strict cloud-only patch.


In [1]:
# ============================================================
# Cell 1. Server/path configuration
# Config-based cloud-only advisor mode.
# IMPORTANT: do not resolve conda/venv Python symlink.
# ============================================================

from pathlib import Path
from datetime import datetime
import os, sys, re, json, time, math, ast, shlex, subprocess, traceback
import pandas as pd
import numpy as np

# Options: "A100_SET_B", "A6000_SET_A"
SERVER_PRESET = os.environ.get("JOILANG_SERVER_PRESET", "A6000_SET_A")

RUN_CLOUDLESS_SMOKE = False
RUN_HYBRID_SMOKE = False

# This notebook runs cloud-only-by-config, not strict cloud-only flags.
RUN_CLOUD_ONLY_SMOKE = True
RUN_FULL_CLOUD_ONLY = False
RUN_FULL_THREE_WAY = False
STRICT_CLOUD_ONLY_REQUIRED = False

SMOKE_MODEL_KEY = os.environ.get("JOILANG_SMOKE_MODEL_KEY", "qwen25_coder_14b")
SMOKE_CATEGORIES = (3, 4, 5, 6)
SMOKE_LIMIT_PER_CATEGORY = 1
SMOKE_SAMPLE_SIZE = 4
SMOKE_VALIDATION_SIZE = 4
SMOKE_POPULATION = 5
SMOKE_GENS = 5

MODEL_LIST_3 = [
    ("7B", "qwen25_coder_7b"),
    ("8B", "llama31_8b"),
    ("14B", "qwen25_coder_14b"),
]

SERVER_CONFIGS = {
    "A100_SET_B": {
        "repo": Path("/root/llm/JOILang-Server"),
        "python": Path("/root/llm/je/bin/python"),  # do not resolve
        "local_model_base": Path("/root/llm/local_models"),
    },
    "A6000_SET_A": {
        "repo": Path("/home/mgjeong/Desktop/llm/JOILang-Server"),
        "python": Path("/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10"),
        "local_model_base": Path("/home/mgjeong/Desktop/llm/local_models"),
    },
}

if SERVER_PRESET not in SERVER_CONFIGS:
    raise ValueError(f"Unknown SERVER_PRESET={SERVER_PRESET}")

cfg = SERVER_CONFIGS[SERVER_PRESET]
REPO = cfg["repo"]             # keep actual server path
PYTHON = cfg["python"]         # intentionally not resolved
py_path = str(PYTHON)
LOCAL_MODEL_BASE = cfg["local_model_base"]

VERSION_DIR = REPO / "gpt_mg/version0_15_update20260413"
SCRIPT = VERSION_DIR / "scripts/run_ga_search.py"
SCRIPTS_DIR = VERSION_DIR / "scripts"
TESTS_DIR = VERSION_DIR / "tests"
RESULTS_ROOT = VERSION_DIR / "results"
NOTEBOOK_DIR = VERSION_DIR / "notebooks"
PAPER_ARTIFACT_ROOT = RESULTS_ROOT / "paper_artifacts"
SUMMARY_DIR = PAPER_ARTIFACT_ROOT / "summary"
FIGURE_DIR = PAPER_ARTIFACT_ROOT / "figures"
TABLE_DIR = PAPER_ARTIFACT_ROOT / "tables"

for d in [RESULTS_ROOT, NOTEBOOK_DIR, PAPER_ARTIFACT_ROOT, SUMMARY_DIR, FIGURE_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_DIRS = {
    "qwen25_coder_7b": "qwen25_coder_7b",
    "llama31_8b": "llama31_8b",
    "qwen25_coder_14b": "qwen25_coder_14b",
    "phi35_mini": "phi35_mini",
    "gemma2_9b_it": "gemma2_9b_it",
}

os.environ["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
os.environ["TRANSFORMERS_OFFLINE"] = os.environ.get("TRANSFORMERS_OFFLINE", "1")
os.environ["HF_HUB_OFFLINE"] = os.environ.get("HF_HUB_OFFLINE", "1")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("=" * 120)
print("[CONFIG]")
print("=" * 120)

path_items = {
    "REPO": REPO,
    "PYTHON": PYTHON,
    "LOCAL_MODEL_BASE": LOCAL_MODEL_BASE,
    "VERSION_DIR": VERSION_DIR,
    "SCRIPT": SCRIPT,
    "RESULTS_ROOT": RESULTS_ROOT,
    "SUMMARY_DIR": SUMMARY_DIR,
}

print("SERVER_PRESET:", SERVER_PRESET)
for k, v in path_items.items():
    print(f"{k}: {v} exists={Path(v).exists()}")

assert REPO.exists(), REPO
assert PYTHON.exists(), PYTHON
assert LOCAL_MODEL_BASE.exists(), LOCAL_MODEL_BASE
assert VERSION_DIR.exists(), VERSION_DIR
assert SCRIPT.exists(), SCRIPT

print("\n[MODE]")
print("Cloud-only-by-config is enabled by default.")
print("Strict cloud-only source flags are optional and are not required for this notebook.")


[CONFIG]
SERVER_PRESET: A6000_SET_A
REPO: /home/mgjeong/Desktop/llm/JOILang-Server exists=True
PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10 exists=True
LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/local_models exists=True
VERSION_DIR: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413 exists=True
SCRIPT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py exists=True
RESULTS_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results exists=True
SUMMARY_DIR: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/paper_artifacts/summary exists=True

[MODE]
Cloud-only-by-config is enabled by default.
Strict cloud-only source flags are optional and are not required for this notebook.


In [2]:
# ============================================================
# Cell 2. Environment verification
# ============================================================

def run_cmd(cmd, timeout=180, check=False, env=None):
    print("\nRUN:", " ".join(map(str, cmd)))
    p = subprocess.run(
        list(map(str, cmd)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=timeout,
        check=False,
        env=env,
    )
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed rc={p.returncode}")
    return p

env_code = r"""
import sys
print("python:", sys.executable)
try:
    import torch
    print("torch:", getattr(torch, "__version__", "NO_VERSION_ATTR"))
    print("cuda:", torch.cuda.is_available())
    print("device_count:", torch.cuda.device_count())
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        print("device0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch_error:", repr(e))
try:
    import transformers
    print("transformers:", getattr(transformers, "__version__", "NO_VERSION_ATTR"))
except Exception as e:
    print("transformers_error:", repr(e))
"""

print("=" * 120)
print("[PYTHON ENV]")
print("=" * 120)
run_cmd([PYTHON, "-c", env_code], timeout=180, check=True)

print("=" * 120)
print("[NVIDIA-SMI]")
print("=" * 120)
run_cmd(["nvidia-smi"], timeout=60, check=False)

print("=" * 120)
print("[OPENAI API KEY]")
print("=" * 120)
print("OPENAI_API_KEY exists:", bool(os.environ.get("OPENAI_API_KEY", "").strip()))

print("=" * 120)
print("[MODEL PATH CHECK]")
print("=" * 120)
rows=[]
for model_key, dirname in MODEL_DIRS.items():
    p = LOCAL_MODEL_BASE / dirname
    rows.append({
        "model_key": model_key,
        "path": str(p),
        "exists": p.exists(),
        "config": (p/"config.json").exists(),
        "tokenizer": (p/"tokenizer.json").exists(),
        "index": (p/"model.safetensors.index.json").exists(),
        "safetensors_count": len(list(p.glob("*.safetensors"))) if p.exists() else 0,
    })
display(pd.DataFrame(rows))

[PYTHON ENV]

RUN: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10 -c 
import sys
print("python:", sys.executable)
try:
    import torch
    print("torch:", getattr(torch, "__version__", "NO_VERSION_ATTR"))
    print("cuda:", torch.cuda.is_available())
    print("device_count:", torch.cuda.device_count())
    if torch.cuda.is_available() and torch.cuda.device_count() > 0:
        print("device0:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch_error:", repr(e))
try:
    import transformers
    print("transformers:", getattr(transformers, "__version__", "NO_VERSION_ATTR"))
except Exception as e:
    print("transformers_error:", repr(e))

python: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
torch: 2.9.1+cu128
cuda: True
device_count: 2
device0: NVIDIA RTX A6000
transformers: 4.57.3

[NVIDIA-SMI]

RUN: nvidia-smi
Fri Jun 12 13:42:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 

,model_key,path,exists,config,tokenizer,index,safetensors_count
0,qwen25_coder_7b,/home/mgjeong/Desktop/llm/local_models/qwen25_...,True,True,True,True,4
1,llama31_8b,/home/mgjeong/Desktop/llm/local_models/llama31_8b,True,True,True,True,4
2,qwen25_coder_14b,/home/mgjeong/Desktop/llm/local_models/qwen25_...,True,True,True,True,6
3,phi35_mini,/home/mgjeong/Desktop/llm/local_models/phi35_mini,False,False,False,False,0
4,gemma2_9b_it,/home/mgjeong/Desktop/llm/local_models/gemma2_...,False,False,False,False,0


In [3]:
# ============================================================
# Cell 3. Source/flag verification
# Config-based cloud-only advisor mode.
# Strict cloud-only flags are optional, not required.
# ============================================================

run_ga_src = SCRIPT.read_text(encoding="utf-8", errors="replace")
advisor_path = SCRIPTS_DIR / "advisor_feedback.py"
advisor_src = advisor_path.read_text(encoding="utf-8", errors="replace") if advisor_path.exists() else ""

help_text = ""
try:
    p = subprocess.run([str(PYTHON), str(SCRIPT), "--help"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=180)
    help_text = p.stdout
    print("help rc:", p.returncode)
except Exception as e:
    print("[WARN] help failed:", repr(e))

SUPPORTED_FLAGS = set(re.findall(r"--[A-Za-z0-9][A-Za-z0-9_-]*", help_text + "\n" + run_ga_src))
print("supported flag count:", len(SUPPORTED_FLAGS))

REQUIRED_BY_CONFIG_FLAGS = [
    "--llm-mutation-advisor",
    "--advisor-model-key",
    "--advisor-trigger-mode",
    "--advisor-force-child-quota",
    "--advisor-min-population-for-child",
    "--advisor-compression-child-quota",
    "--advisor-prefer-compression-after-detpass",

    "--enable-compression-mutation",
    "--compression-detpass-threshold",
    "--aggressive-compression-after-target",
    "--compression-child-quota",
    "--compression-child-ratio",
    "--compression-token-reduction-target",
    "--compression-token-plateau-delta",
    "--allow-aggressive-compression",

    "--micro-compression-child-quota",
    "--micro-compression-child-ratio",
    "--block-compression-child-quota",
    "--block-compression-child-ratio",
    "--multi-block-compression-child-quota",
    "--multi-block-compression-child-ratio",
    "--global-budget-compression-child-quota",

    "--enable-block-token-breakdown",
    "--enable-multi-block-compression",
    "--enable-render-budget-compression",
    "--min-compression-token-delta",

    "--mutation-mode",
    "--selection-mode",
    "--fitness-mode",
    "--token-penalty-mode",
    "--stop-controller-mode",
    "--reasoning-mutation-mode",
    "--intent-hint-mode",
]

OPTIONAL_STRICT_CLOUD_ONLY_FLAGS = [
    "--cloud-only-advisor-compression",
    "--disable-compression-fallback",
    "--advisor-proposal-k",
    "--advisor-min-usable-compression-proposals",
    "--advisor-repair-invalid-proposals",
    "--advisor-repair-max-attempts",
    "--advisor-require-block-proposal-after-detpass",
    "--advisor-require-token-delta",
    "--advisor-disallow-genome-only-after-detpass",
    "--advisor-cloud-only-strict-schema",
    "--advisor-before-after-report",
    "--advisor-include-dataset-feedback",
    "--advisor-feedback-topk",
    "--advisor-feedback-max-failures-per-family",
    "--advisor-include-prompt-before-after-context",
    "--advisor-include-case-abc-sections",
    "--advisor-write-debug-prompt-sections",
]

OPTIONAL_STRICT_SECTIONS = [
    "DATASET_EVALUATION_FEEDBACK",
    "CASE_A_MICRO_COMPRESSION",
    "CASE_B_THRESHOLD_BLOCK_COMPRESSION",
    "CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION",
    "STRICT_JSON_RESPONSE_SCHEMA",
    "PROMPT_TOKEN_BREAKDOWN",
    "BLOCK_TOKEN_BREAKDOWN",
]

EXISTING_OBSERVABILITY_STRINGS = [
    "compression_policy",
    "block_token_breakdown",
    "prompt_token_breakdown",
    "advisor_prompt_generation",
    "advisor_response_generation",
    "advisor_mutation_proposals",
    "mutation_proposals",
    "population_transitions",
]

rows = []
for f in REQUIRED_BY_CONFIG_FLAGS:
    rows.append({
        "kind": "required_by_config_flag",
        "name": f,
        "present": f in SUPPORTED_FLAGS or f in run_ga_src,
        "required_for_by_config": True,
    })

for f in OPTIONAL_STRICT_CLOUD_ONLY_FLAGS:
    rows.append({
        "kind": "optional_strict_cloud_only_flag",
        "name": f,
        "present": f in SUPPORTED_FLAGS or f in run_ga_src,
        "required_for_by_config": False,
    })

for s in EXISTING_OBSERVABILITY_STRINGS:
    rows.append({
        "kind": "existing_observability_string",
        "name": s,
        "present": (s in run_ga_src) or (s in advisor_src),
        "required_for_by_config": False,
    })

for s in OPTIONAL_STRICT_SECTIONS:
    rows.append({
        "kind": "optional_strict_prompt_section",
        "name": s,
        "present": (s in advisor_src) or (s in run_ga_src),
        "required_for_by_config": False,
    })

verify_df = pd.DataFrame(rows)
display(verify_df)

missing_required = verify_df[
    (verify_df["required_for_by_config"] == True)
    & (verify_df["present"] == False)
]["name"].tolist()

STRICT_CLOUD_ONLY_IMPLEMENTED = bool(
    verify_df[
        verify_df["kind"].isin(["optional_strict_cloud_only_flag", "optional_strict_prompt_section"])
    ]["present"].all()
)

print("=" * 120)
print("[SOURCE / FLAG VERIFICATION SUMMARY]")
print("=" * 120)
print("STRICT_CLOUD_ONLY_IMPLEMENTED:", STRICT_CLOUD_ONLY_IMPLEMENTED)

if missing_required:
    print("[ERROR] Missing flags required for cloud-only-by-config:")
    for x in missing_required:
        print("-", x)
    raise RuntimeError("Existing strong compression flags are missing; cannot run cloud-only-by-config.")

print("[OK] Existing strong compression flags are available.")
print("[MODE] This notebook will run cloud-only-by-config.")
print("[NOTE] Strict cloud-only flags/sections are optional and currently:",
      "implemented" if STRICT_CLOUD_ONLY_IMPLEMENTED else "not implemented")


help rc: 0
supported flag count: 134


,kind,name,present,required_for_by_config
0,required_by_config_flag,--llm-mutation-advisor,True,True
1,required_by_config_flag,--advisor-model-key,True,True
2,required_by_config_flag,--advisor-trigger-mode,True,True
3,required_by_config_flag,--advisor-force-child-quota,True,True
4,required_by_config_flag,--advisor-min-population-for-child,True,True
...,...,...,...,...
60,optional_strict_prompt_section,CASE_B_THRESHOLD_BLOCK_COMPRESSION,False,False
61,optional_strict_prompt_section,CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION,False,False
62,optional_strict_prompt_section,STRICT_JSON_RESPONSE_SCHEMA,False,False
63,optional_strict_prompt_section,PROMPT_TOKEN_BREAKDOWN,False,False


[SOURCE / FLAG VERIFICATION SUMMARY]
STRICT_CLOUD_ONLY_IMPLEMENTED: False
[OK] Existing strong compression flags are available.
[MODE] This notebook will run cloud-only-by-config.
[NOTE] Strict cloud-only flags/sections are optional and currently: not implemented


In [4]:
# ============================================================
# Cell 4. run_ga_all_categories wrapper
# ============================================================

def flag_supported(flag: str) -> bool:
    return flag in SUPPORTED_FLAGS or flag in run_ga_src

def add_value(cmd, flag, value, *, required=False):
    if value is None:
        return
    if not flag_supported(flag):
        if required:
            raise RuntimeError(f"Required unsupported flag: {flag}")
        print("[SKIP unsupported flag]", flag)
        return
    cmd.extend([flag, str(value)])

def add_bool(cmd, flag, enabled, *, required=False):
    if not enabled:
        return
    if not flag_supported(flag):
        if required:
            raise RuntimeError(f"Required unsupported flag: {flag}")
        print("[SKIP unsupported flag]", flag)
        return
    cmd.append(flag)

def model_env_name(model_key):
    p = LOCAL_MODEL_BASE / MODEL_DIRS.get(model_key, model_key)
    return str(p) if p.exists() else ""

def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def run_ga_all_categories(
    model_key="qwen25_coder_14b",
    categories=(1,),
    limit_per_category=1,
    sample_size=1,
    validation_size=1,
    population=4,
    gens=3,
    target_detpass=90,
    base_prefix=None,
    use_advisor=False,
    full_run=True,
    progress="verbose",
    timeout_sec=1200,
    retries=0,
    advisor_trigger_mode="always",
    advisor_min_population_for_child=4,
    advisor_force_child_quota=True,
    use_mock_advisor=False,
    advisor_model_key="gpt41_mini",

    compression_detpass_threshold=90,
    aggressive_compression_after_target=True,
    allow_aggressive_compression=True,
    compression_child_quota=1,
    compression_child_ratio=0.25,
    micro_compression_child_quota=1,
    micro_compression_child_ratio=0.25,
    block_compression_child_quota=1,
    block_compression_child_ratio=0.25,
    multi_block_compression_child_quota=1,
    multi_block_compression_child_ratio=0.25,
    global_budget_compression_child_quota=0,
    advisor_compression_child_quota=1,
    advisor_prefer_compression_after_detpass=90,
    compression_token_reduction_target=0.15,
    compression_token_plateau_delta=1.0,
    enable_block_token_breakdown=True,
    enable_multi_block_compression=True,
    enable_render_budget_compression=False,
    min_compression_token_delta=50,

    cloud_only_advisor_compression=False,
    disable_compression_fallback=False,
    advisor_proposal_k=6,
    advisor_min_usable_compression_proposals=2,
    advisor_repair_invalid_proposals=False,
    advisor_repair_max_attempts=0,
    advisor_require_block_proposal_after_detpass=False,
    advisor_require_token_delta=False,
    advisor_disallow_genome_only_after_detpass=False,
    advisor_cloud_only_strict_schema=False,
    advisor_before_after_report=False,
    advisor_include_dataset_feedback=False,
    advisor_feedback_topk=5,
    advisor_feedback_max_failures_per_family=3,
    advisor_include_prompt_before_after_context=False,
    advisor_include_case_abc_sections=False,
    advisor_write_debug_prompt_sections=False,
):
    if cloud_only_advisor_compression and not use_advisor:
        raise ValueError("cloud_only_advisor_compression requires use_advisor=True")

    is_cloud_only_by_config = bool(
        use_advisor
        and not cloud_only_advisor_compression
        and float(compression_child_quota or 0) == 0
        and float(compression_child_ratio or 0.0) == 0.0
        and float(micro_compression_child_quota or 0) == 0
        and float(micro_compression_child_ratio or 0.0) == 0.0
        and float(block_compression_child_quota or 0) == 0
        and float(block_compression_child_ratio or 0.0) == 0.0
        and float(multi_block_compression_child_quota or 0) == 0
        and float(multi_block_compression_child_ratio or 0.0) == 0.0
        and float(global_budget_compression_child_quota or 0) == 0
        and float(advisor_compression_child_quota or 0) > 0
    )
    mode_name = "cloud_only_by_config" if is_cloud_only_by_config else ("cloud_only_advisor" if cloud_only_advisor_compression else ("cloud_advisor" if use_advisor else "cloudless"))
    base_prefix = base_prefix or f"smoke_{SERVER_PRESET}_{mode_name}"
    cat_label = "".join(map(str, categories))

    run_dir = RESULTS_ROOT / f"{base_prefix}_{mode_name}_cat{cat_label}_lpc{limit_per_category}_pop{population}_gens{gens}_{model_key}_{timestamp()}"
    output_root = run_dir / "ga_output"
    output_root.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(PYTHON), "-u", str(SCRIPT),
        "--profile", "version0_15",
        "--model-key", model_key,
        "--target-detpass", str(target_detpass),
        "--llm-mode", "worker",
        "--population", str(population),
        "--gens", str(gens),
        "--min-generations", str(gens),
        "--max-generations", str(gens),
        "--sample-size", str(sample_size),
        "--validation-size", str(validation_size),
        "--cheap-eval-limit", "2",
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--selection-mode", "redesign",
        "--fitness-mode", "phase_aware",
        "--mutation-mode", "cloudless_decompiler",
        "--category-balance-mode", "guard",
        "--token-penalty-mode", "hybrid",
        "--stop-controller-mode", "active",
        "--plateau-window", "1",
        "--disruptive-max-attempts", "1",
        "--reasoning-mutation-mode", "auto",
        "--intent-hint-mode", "auto",
        "--progress", progress,
        "--timeout-sec", str(timeout_sec),
        "--retries", str(retries),
        "--limit-per-category", str(limit_per_category),
        "--output-root", str(output_root),
    ]

    if full_run:
        cmd.append("--full-run")
    for flag in [
        "--feedback-guided-mutation",
        "--enable-compression-mutation",
        "--enable-prompt-decompiler",
        "--enable-rendered-prompt-dedupe",
        "--enable-pareto-archive",
        "--enable-group-specialist-archives",
    ]:
        if flag_supported(flag):
            cmd.append(flag)

    for c in categories:
        cmd.extend(["--category", str(c)])

    add_value(cmd, "--compression-detpass-threshold", compression_detpass_threshold)
    add_bool(cmd, "--aggressive-compression-after-target", aggressive_compression_after_target)
    add_bool(cmd, "--allow-aggressive-compression", allow_aggressive_compression)
    add_value(cmd, "--compression-child-quota", compression_child_quota)
    add_value(cmd, "--compression-child-ratio", compression_child_ratio)
    add_value(cmd, "--micro-compression-child-quota", micro_compression_child_quota)
    add_value(cmd, "--micro-compression-child-ratio", micro_compression_child_ratio)
    add_value(cmd, "--block-compression-child-quota", block_compression_child_quota)
    add_value(cmd, "--block-compression-child-ratio", block_compression_child_ratio)
    add_value(cmd, "--multi-block-compression-child-quota", multi_block_compression_child_quota)
    add_value(cmd, "--multi-block-compression-child-ratio", multi_block_compression_child_ratio)
    add_value(cmd, "--global-budget-compression-child-quota", global_budget_compression_child_quota)
    add_value(cmd, "--advisor-compression-child-quota", advisor_compression_child_quota)
    add_value(cmd, "--advisor-prefer-compression-after-detpass", advisor_prefer_compression_after_detpass)
    add_value(cmd, "--compression-token-reduction-target", compression_token_reduction_target)
    add_value(cmd, "--compression-token-plateau-delta", compression_token_plateau_delta)
    add_bool(cmd, "--enable-block-token-breakdown", enable_block_token_breakdown)
    add_bool(cmd, "--enable-multi-block-compression", enable_multi_block_compression)
    add_bool(cmd, "--enable-render-budget-compression", enable_render_budget_compression)
    add_value(cmd, "--min-compression-token-delta", min_compression_token_delta)

    if use_advisor:
        cmd.append("--llm-mutation-advisor")
        add_value(cmd, "--advisor-model-key", advisor_model_key)
        add_value(cmd, "--advisor-trigger-mode", advisor_trigger_mode)
        add_value(cmd, "--advisor-min-population-for-child", advisor_min_population_for_child)
        add_bool(cmd, "--advisor-force-child-quota", advisor_force_child_quota)
        add_bool(cmd, "--use-mock-advisor", use_mock_advisor)

        req = bool(cloud_only_advisor_compression)
        add_bool(cmd, "--cloud-only-advisor-compression", cloud_only_advisor_compression, required=req)
        add_bool(cmd, "--disable-compression-fallback", disable_compression_fallback, required=req)
        add_value(cmd, "--advisor-proposal-k", advisor_proposal_k, required=req)
        add_value(cmd, "--advisor-min-usable-compression-proposals", advisor_min_usable_compression_proposals, required=req)
        add_bool(cmd, "--advisor-repair-invalid-proposals", advisor_repair_invalid_proposals, required=req)
        add_value(cmd, "--advisor-repair-max-attempts", advisor_repair_max_attempts, required=req)
        add_bool(cmd, "--advisor-require-block-proposal-after-detpass", advisor_require_block_proposal_after_detpass, required=req)
        add_bool(cmd, "--advisor-require-token-delta", advisor_require_token_delta, required=req)
        add_bool(cmd, "--advisor-disallow-genome-only-after-detpass", advisor_disallow_genome_only_after_detpass, required=req)
        add_bool(cmd, "--advisor-cloud-only-strict-schema", advisor_cloud_only_strict_schema, required=req)
        add_bool(cmd, "--advisor-before-after-report", advisor_before_after_report, required=req)
        add_bool(cmd, "--advisor-include-dataset-feedback", advisor_include_dataset_feedback, required=req)
        add_value(cmd, "--advisor-feedback-topk", advisor_feedback_topk, required=req)
        add_value(cmd, "--advisor-feedback-max-failures-per-family", advisor_feedback_max_failures_per_family, required=req)
        add_bool(cmd, "--advisor-include-prompt-before-after-context", advisor_include_prompt_before_after_context, required=req)
        add_bool(cmd, "--advisor-include-case-abc-sections", advisor_include_case_abc_sections, required=req)
        add_bool(cmd, "--advisor-write-debug-prompt-sections", advisor_write_debug_prompt_sections, required=req)
    else:
        add_value(cmd, "--advisor-trigger-mode", "off")

    env = os.environ.copy()
    env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
    model_name = model_env_name(model_key)
    if model_name:
        env["JOI_V15_LOCAL_MODEL_NAME"] = model_name
    env["JOI_V15_LOCAL_FILES_ONLY"] = "true"
    env["TRANSFORMERS_OFFLINE"] = env.get("TRANSFORMERS_OFFLINE", "1")
    env["HF_HUB_OFFLINE"] = env.get("HF_HUB_OFFLINE", "1")
    env["TOKENIZERS_PARALLELISM"] = "false"
    debug_log = Path("/tmp") / f"joi_v15_worker_debug_{model_key}_{mode_name}_{timestamp()}.log"
    env["JOI_V15_WORKER_DEBUG_LOG"] = str(debug_log)

    print("\n" + "=" * 120)
    print(f"RUN: {model_key} / {mode_name}")
    print("OUTPUT:", output_root)
    print("PYTHON:", PYTHON)
    print("MODEL_BASE:", LOCAL_MODEL_BASE)
    print("MODEL_NAME:", env.get("JOI_V15_LOCAL_MODEL_NAME", "<run_ga_search default>"))
    print("DEBUG_LOG:", debug_log)
    print("=" * 120)
    print("COMMAND:")
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 120)

    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    rc = proc.wait()

    (output_root / "_notebook_command.txt").write_text(" ".join(shlex.quote(str(x)) for x in cmd), encoding="utf-8")
    (output_root / "_notebook_stdout_tail.txt").write_text("".join(lines[-2000:]), encoding="utf-8")

    print("\nRETURN CODE:", rc)
    print("OUTPUT:", output_root)
    if debug_log.exists():
        print("DEBUG_LOG:", debug_log)
    if rc != 0:
        raise RuntimeError(f"run_ga_all_categories failed with return code {rc}")
    return output_root

print("[OK] wrapper defined")


[OK] wrapper defined


In [11]:
# ============================================================
# Cell 5. Artifact readers and analysis helpers
# ============================================================

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8", errors="replace"))
    except Exception as e:
        print("[JSON READ ERROR]", path, repr(e))
        return None

def read_jsonl(path):
    rows = []
    path = Path(path)
    if not path.exists():
        return rows
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            rows.append({"_raw": line, "_parse_error": True})
    return rows

def safe_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("[CSV READ ERROR]", path, repr(e))
        return pd.DataFrame()

def extract_blocks(obj):
    if obj is None:
        return []
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict):
        for k in ["blocks", "block_token_breakdown", "block_breakdown", "items", "rows", "block_rows"]:
            if isinstance(obj.get(k), list):
                return obj[k]
        rows = []
        for k, v in obj.items():
            if isinstance(v, dict):
                vv = dict(v)
                vv.setdefault("block_id", k)
                rows.append(vv)
        return rows
    return []

def classify_case_abc(row):
    level = str(row.get("compression_level", "") or "").lower()
    op = str(row.get("operator", row.get("mutation_type", row.get("mutation", ""))) or "").lower()
    target = str(row.get("target_block_id", row.get("selected_block_id", "")) or "").lower()
    ids = row.get("selected_block_ids", [])
    if isinstance(ids, str):
        try:
            ids = ast.literal_eval(ids)
        except Exception:
            ids = []
    if level in {"multi_block", "global_budget", "global", "render_budget"}:
        return "Case C"
    if "multi" in op or "global" in op or "budget" in op:
        return "Case C"
    if isinstance(ids, list) and len(ids) >= 2:
        return "Case C"
    if level == "block":
        return "Case B"
    if target and target not in {"genome", "none", "nan", ""}:
        return "Case B"
    if "drop_optional_block" in op or "compact_reasoning_skeleton" in op:
        return "Case B"
    if level == "micro":
        return "Case A"
    if any(k in op for k in ["candidate_strategies", "lower_output_max_tokens", "template_compress", "dedupe", "safe"]):
        return "Case A"
    if any(k in op for k in ["few_shot", "micro_rules", "compact_block_params"]):
        return "Case B"
    return "Unclassified"

def load_progress(out_dir):
    return safe_csv(Path(out_dir) / "ga_generation_progress.csv")

def load_transitions(out_dir):
    return safe_csv(Path(out_dir) / "population_transitions.csv")

def load_all_proposals(out_dir):
    out_dir = Path(out_dir)
    rows = []
    for fn in ["advisor_mutation_proposals.jsonl", "mutation_proposals.jsonl"]:
        for r in read_jsonl(out_dir / fn):
            rr = dict(r)
            rr["_file"] = fn
            if "operator" not in rr:
                rr["operator"] = rr.get("mutation_type", rr.get("mutation", ""))
            rr["case_label_inferred"] = classify_case_abc(rr)
            rows.append(rr)
    return pd.DataFrame(rows)

def summarize_run(out_dir, label=None, mode=None):
    out_dir = Path(out_dir)
    row = {
        "label": label or out_dir.parent.name,
        "mode": mode,
        "out_dir": str(out_dir),
        "exists": out_dir.exists(),
        "summary_exists": (out_dir/"ga_summary.json").exists(),
        "progress_exists": (out_dir/"ga_generation_progress.csv").exists(),
        "transitions_exists": (out_dir/"population_transitions.csv").exists(),
        "block_breakdown_exists": (out_dir/"block_token_breakdown.json").exists(),
        "prompt_breakdown_exists": (out_dir/"prompt_token_breakdown.json").exists(),
    }
    s = read_json(out_dir/"ga_summary.json") or {}
    row["best_DETPass"] = s.get("best_DETPass", s.get("best_so_far_DETPass", np.nan))
    row["advisor_model_key"] = s.get("advisor_model_key")
    p = load_progress(out_dir)
    if len(p):
        for c in ["validation_det_pass_rate", "validation_avg_det_score", "best_so_far_DETPass"]:
            if c in p.columns:
                row[c] = float(pd.to_numeric(p[c], errors="coerce").fillna(0).max())
        if "avg_prompt_tokens" in p.columns:
            toks = pd.to_numeric(p["avg_prompt_tokens"], errors="coerce").dropna()
            if len(toks):
                row["first_tokens"] = float(toks.iloc[0])
                row["last_tokens"] = float(toks.iloc[-1])
                row["min_tokens"] = float(toks.min())
                row["token_delta_last"] = row["last_tokens"] - row["first_tokens"]
                row["token_reduction_ratio_min"] = (row["first_tokens"] - row["min_tokens"]) / row["first_tokens"] if row["first_tokens"] else np.nan
    t = load_transitions(out_dir)
    for col in [
        "new_by_compression", "new_by_micro_compression", "new_by_block_compression",
        "new_by_multi_block_compression", "new_by_global_budget_compression",
        "new_by_compression_fallback", "new_by_advisor",
        "advisor_proposals_generated", "advisor_proposals_accepted_applied",
        "advisor_proposals_rejected", "advisor_unfilled_quota",
    ]:
        if len(t) and col in t.columns:
            row[col + "_sum"] = float(pd.to_numeric(t[col], errors="coerce").fillna(0).sum())
    prop = load_all_proposals(out_dir)
    row["proposal_rows"] = len(prop)
    row["fallback_rows"] = int((prop.get("source", pd.Series([], dtype=str)).astype(str) == "compression_fallback").sum()) if len(prop) else 0
    row["advisor_accepted_rows"] = int(((prop.get("_file", pd.Series([], dtype=str)) == "advisor_mutation_proposals.jsonl") & (prop.get("accepted", pd.Series([], dtype=bool)) == True)).sum()) if len(prop) else 0
    row["advisor_rejected_rows"] = int(((prop.get("_file", pd.Series([], dtype=str)) == "advisor_mutation_proposals.jsonl") & (prop.get("accepted", pd.Series([], dtype=bool)) == False)).sum()) if len(prop) else 0
    return row

def prompt_keyword_table(out_dir):
    out_dir = Path(out_dir)
    keys = [
        "ADVISOR_ROLE", "CURRENT_STATE", "DATASET_EVALUATION_FEEDBACK",
        "PROMPT_TOKEN_BREAKDOWN", "BLOCK_TOKEN_BREAKDOWN",
        "CASE_A_MICRO_COMPRESSION", "CASE_B_THRESHOLD_BLOCK_COMPRESSION",
        "CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION", "STRICT_JSON_RESPONSE_SCHEMA",
    ]
    rows = []
    for p in sorted(out_dir.glob("advisor_prompt_generation_*.txt")):
        txt = p.read_text(encoding="utf-8", errors="replace")
        row = {"file": p.name, "chars": len(txt), "approx_tokens": round(len(txt)/4)}
        for k in keys:
            row[k] = k in txt
        rows.append(row)
    return pd.DataFrame(rows)

def inspect_run(out_dir, max_rows=30):
    out_dir = Path(out_dir)
    print("\n" + "#"*120)
    print("INSPECT RUN:", out_dir)
    print("#"*120)
    files = [
        "ga_summary.json", "ga_generation_progress.csv", "population_transitions.csv",
        "mutation_proposals.jsonl", "advisor_mutation_proposals.jsonl",
        "advisor_mutation_summary.csv", "advisor_cloud_only_summary.csv",
        "advisor_cloud_only_before_after.csv", "advisor_case_ABC_summary.csv",
        "block_token_breakdown.json", "prompt_token_breakdown.json",
    ]
    for f in files:
        print(f"{f:45s}", (out_dir/f).exists())
    print("\n[SUMMARY]")
    display(pd.DataFrame([summarize_run(out_dir)]))
    p = load_progress(out_dir)
    if len(p):
        cols = [c for c in ["generation","validation_det_pass_rate","validation_avg_det_score","best_so_far_DETPass","avg_prompt_tokens","compression_ready","compression_phase","generation_phase","next_action"] if c in p.columns]
        print("\n[GA PROGRESS]")
        display(p[cols].tail(max_rows))
    t = load_transitions(out_dir)
    if len(t):
        cols = [c for c in ["generation","fallback_disabled","new_by_compression","new_by_micro_compression","new_by_block_compression","new_by_multi_block_compression","new_by_compression_fallback","new_by_advisor","advisor_proposals_generated","advisor_proposals_accepted_applied","advisor_proposals_rejected","advisor_unfilled_quota"] if c in t.columns]
        print("\n[TRANSITIONS]")
        display(t[cols].tail(max_rows))
    print("\n[ADVISOR PROMPT KEYWORDS]")
    df = prompt_keyword_table(out_dir)
    display(df if len(df) else pd.DataFrame([{"message":"No advisor prompt files"}]))
    print("\n[PROPOSALS]")
    prop = load_all_proposals(out_dir)
    if len(prop):
        cols = [c for c in ["_file","generation","source","schema_source","case_label","case_label_inferred","operator","compression_level","selected_block_id","selected_block_ids","expected_token_delta","measured_prompt_token_delta","accepted","applied","rejection_reason","fallback_reason","dataset_feedback_reason"] if c in prop.columns]
        display(prop[cols].tail(max_rows))
    else:
        print("No proposal rows.")

print("[OK] helpers defined")

In [6]:
# ============================================================
# Cell 6. Cloudless baseline smoke
# ============================================================

if RUN_CLOUDLESS_SMOKE:
    cloudless_out = run_ga_all_categories(
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=SMOKE_GENS,
        base_prefix=f"smoke_cloudless_baseline_{SERVER_PRESET}",
        use_advisor=False,
        timeout_sec=2400,
        compression_child_quota=1,
        compression_child_ratio=0.25,
        micro_compression_child_quota=1,
        micro_compression_child_ratio=0.25,
        block_compression_child_quota=1,
        block_compression_child_ratio=0.25,
        multi_block_compression_child_quota=1,
        multi_block_compression_child_ratio=0.25,
        advisor_compression_child_quota=0,
    )
    inspect_run(cloudless_out)
else:
    print("[SKIP] RUN_CLOUDLESS_SMOKE=False. This A6000 notebook is cloud-only by default.")

[SKIP] RUN_CLOUDLESS_SMOKE=False. This A6000 notebook is cloud-only by default.


In [7]:
# ============================================================
# Cell 7. Hybrid advisor smoke
# ============================================================

if RUN_HYBRID_SMOKE:
    hybrid_out = run_ga_all_categories(
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=SMOKE_GENS,
        base_prefix=f"smoke_hybrid_advisor_{SERVER_PRESET}",
        use_advisor=True,
        timeout_sec=2400,
        advisor_model_key="gpt41_mini",
        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,
        compression_child_quota=1,
        compression_child_ratio=0.25,
        micro_compression_child_quota=1,
        micro_compression_child_ratio=0.25,
        block_compression_child_quota=1,
        block_compression_child_ratio=0.25,
        multi_block_compression_child_quota=1,
        multi_block_compression_child_ratio=0.25,
        advisor_compression_child_quota=1,
        advisor_include_dataset_feedback=True,
        advisor_include_case_abc_sections=True,
        advisor_write_debug_prompt_sections=True,
    )
    inspect_run(hybrid_out)
else:
    print("[SKIP] RUN_HYBRID_SMOKE=False. This A6000 notebook is cloud-only by default.")

[SKIP] RUN_HYBRID_SMOKE=False. This A6000 notebook is cloud-only by default.


In [8]:
# ============================================================
# Cell 8. Cloud-only-by-config advisor smoke
# No strict cloud-only flags are required.
# Local compression quotas/ratios are set to 0; advisor compression quota remains positive.
# The final gate checks whether fallback actually stayed at 0.
# ============================================================

if RUN_CLOUD_ONLY_SMOKE:
    cloud_only_by_config_out = run_ga_all_categories(
        model_key=SMOKE_MODEL_KEY,
        categories=SMOKE_CATEGORIES,
        limit_per_category=SMOKE_LIMIT_PER_CATEGORY,
        sample_size=SMOKE_SAMPLE_SIZE,
        validation_size=SMOKE_VALIDATION_SIZE,
        population=SMOKE_POPULATION,
        gens=SMOKE_GENS,
        base_prefix=f"smoke_cloud_only_by_config_{SERVER_PRESET}",
        use_advisor=True,
        timeout_sec=2400,
        advisor_model_key="gpt41_mini",
        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,

        # Cloud-only-by-config isolation:
        # disable local/fallback compression lanes by quota and ratio.
        cloud_only_advisor_compression=False,
        disable_compression_fallback=False,
        compression_child_quota=0,
        compression_child_ratio=0.0,
        micro_compression_child_quota=0,
        micro_compression_child_ratio=0.0,
        block_compression_child_quota=0,
        block_compression_child_ratio=0.0,
        multi_block_compression_child_quota=0,
        multi_block_compression_child_ratio=0.0,
        global_budget_compression_child_quota=0,

        # Keep only advisor compression child scheduling active.
        advisor_compression_child_quota=2,
        advisor_prefer_compression_after_detpass=90,

        compression_detpass_threshold=90,
        aggressive_compression_after_target=True,
        allow_aggressive_compression=True,
        compression_token_reduction_target=0.15,
        compression_token_plateau_delta=1.0,
        enable_block_token_breakdown=True,
        enable_multi_block_compression=True,
        enable_render_budget_compression=False,
        min_compression_token_delta=50,

        # Strict cloud-only extras are disabled because this repo does not expose those flags.
        advisor_repair_invalid_proposals=False,
        advisor_require_block_proposal_after_detpass=False,
        advisor_require_token_delta=False,
        advisor_disallow_genome_only_after_detpass=False,
        advisor_cloud_only_strict_schema=False,
        advisor_before_after_report=False,
        advisor_include_dataset_feedback=False,
        advisor_include_prompt_before_after_context=False,
        advisor_include_case_abc_sections=False,
        advisor_write_debug_prompt_sections=False,
    )

    # Backward-compatible aliases used by later analysis cells.
    cloud_only_out = cloud_only_by_config_out
    inspect_run(cloud_only_by_config_out)
else:
    raise RuntimeError("RUN_CLOUD_ONLY_SMOKE=False. Set True to run cloud-only-by-config smoke.")


[SKIP unsupported flag] --advisor-proposal-k
[SKIP unsupported flag] --advisor-min-usable-compression-proposals
[SKIP unsupported flag] --advisor-repair-max-attempts
[SKIP unsupported flag] --advisor-feedback-topk
[SKIP unsupported flag] --advisor-feedback-max-failures-per-family

RUN: qwen25_coder_14b / cloud_only_by_config
OUTPUT: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_cloud_only_by_config_A6000_SET_A_cloud_only_by_config_cat3456_lpc1_pop5_gens5_qwen25_coder_14b_20260612_134308/ga_output
PYTHON: /home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10
MODEL_BASE: /home/mgjeong/Desktop/llm/local_models
MODEL_NAME: /home/mgjeong/Desktop/llm/local_models/qwen25_coder_14b
DEBUG_LOG: /tmp/joi_v15_worker_debug_qwen25_coder_14b_cloud_only_by_config_20260612_134308.log
COMMAND:
/home/mgjeong/miniconda3/envs/paper-gpu/bin/python3.10 -u /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_ga_search.py --profile v

,label,mode,out_dir,exists,summary_exists,progress_exists,transitions_exists,block_breakdown_exists,prompt_breakdown_exists,best_DETPass,...,new_by_global_budget_compression_sum,new_by_compression_fallback_sum,new_by_advisor_sum,advisor_proposals_generated_sum,advisor_proposals_accepted_applied_sum,advisor_proposals_rejected_sum,proposal_rows,fallback_rows,advisor_accepted_rows,advisor_rejected_rows
0,smoke_cloud_only_by_config_A6000_SET_A_cloud_o...,None,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_m...,True,True,True,True,True,True,75.0,...,0.0,0.0,0.0,5.0,0.0,5.0,18,0,0,5



[GA PROGRESS]


,generation,validation_det_pass_rate,validation_avg_det_score,best_so_far_DETPass,avg_prompt_tokens,compression_ready,compression_phase,generation_phase,next_action
0,1,75.0,73.4821,75.0,39998.5,False,ACCURACY_SEARCH,ACCURACY_SEARCH,continue_accuracy
1,2,75.0,74.4288,75.0,39241.5,False,ACCURACY_SEARCH,ACCURACY_SEARCH,continue_accuracy
2,3,75.0,74.4288,75.0,39241.5,False,ACCURACY_SEARCH,ACCURACY_SEARCH,continue_accuracy
3,4,75.0,74.4288,75.0,39241.5,False,ACCURACY_SEARCH,ACCURACY_SEARCH,continue_accuracy
4,5,75.0,74.4288,75.0,39241.5,False,ACCURACY_SEARCH,FINAL_SELECTION,stop_and_finalize



[TRANSITIONS]


,generation,new_by_compression,new_by_micro_compression,new_by_block_compression,new_by_multi_block_compression,new_by_compression_fallback,new_by_advisor,advisor_proposals_generated,advisor_proposals_accepted_applied,advisor_proposals_rejected
0,1,1,1,0,0,0,0,1,0,1
1,2,1,1,0,0,0,0,1,0,1
2,3,1,1,0,0,0,0,1,0,1
3,4,1,1,0,0,0,0,1,0,1
4,5,1,1,0,0,0,0,1,0,1



[ADVISOR PROMPT KEYWORDS]


,file,chars,approx_tokens,ADVISOR_ROLE,CURRENT_STATE,DATASET_EVALUATION_FEEDBACK,PROMPT_TOKEN_BREAKDOWN,BLOCK_TOKEN_BREAKDOWN,CASE_A_MICRO_COMPRESSION,CASE_B_THRESHOLD_BLOCK_COMPRESSION,CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION,STRICT_JSON_RESPONSE_SCHEMA
0,advisor_prompt_generation_001.txt,39036,9759,False,False,False,False,False,False,False,False,False
1,advisor_prompt_generation_002.txt,40233,10058,False,False,False,False,False,False,False,False,False
2,advisor_prompt_generation_003.txt,32881,8220,False,False,False,False,False,False,False,False,False
3,advisor_prompt_generation_004.txt,39053,9763,False,False,False,False,False,False,False,False,False
4,advisor_prompt_generation_005.txt,32842,8210,False,False,False,False,False,False,False,False,False



[PROPOSALS]


,_file,generation,source,schema_source,case_label_inferred,operator,compression_level,selected_block_id,selected_block_ids,expected_token_delta,measured_prompt_token_delta,accepted,rejection_reason,fallback_reason
0,advisor_mutation_proposals.jsonl,1,advisor,no_schema,Unclassified,,,,[],0,NaN,False,no_advisor_proposals_parsed,
1,advisor_mutation_proposals.jsonl,2,advisor,no_schema,Unclassified,,,,[],0,NaN,False,no_advisor_proposals_parsed,
2,advisor_mutation_proposals.jsonl,3,advisor,no_schema,Unclassified,,,,[],0,NaN,False,no_advisor_proposals_parsed,
3,advisor_mutation_proposals.jsonl,4,advisor,no_schema,Unclassified,,,,[],0,NaN,False,no_advisor_proposals_parsed,
4,advisor_mutation_proposals.jsonl,5,advisor,no_schema,Unclassified,,,,[],0,NaN,False,no_advisor_proposals_parsed,
5,mutation_proposals.jsonl,2,det_feedback,,Case B,strengthen_json_only_rule,,,[],0,26.0,True,,
6,mutation_proposals.jsonl,2,cloudless,,Case A,reduce_few_shot_count_to_zero,micro,,[],0,-757.0,True,,
7,mutation_proposals.jsonl,1,advisor,no_schema,Unclassified,,,,[],0,NaN,False,no_advisor_proposals_parsed,
8,mutation_proposals.jsonl,3,det_feedback,,Case B,add_targeted_repair_hint,,,[],0,0.0,True,,
9,mutation_proposals.jsonl,3,cloudless,,Case A,merge_duplicate_micro_rules,micro,,[],0,0.0,True,,


### advisor_prompt_generation_001~5.txt 확인: Cell 8이 만든 GA output directory

In [10]:
from pathlib import Path

OUT_DIR = Path(cloud_only_by_config_out)

print("OUT_DIR:", OUT_DIR)
for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt")):
    print(p.name, p)

OUT_DIR: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_cloud_only_by_config_A6000_SET_A_cloud_only_by_config_cat3456_lpc1_pop5_gens5_qwen25_coder_14b_20260612_134308/ga_output
advisor_prompt_generation_001.txt /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_cloud_only_by_config_A6000_SET_A_cloud_only_by_config_cat3456_lpc1_pop5_gens5_qwen25_coder_14b_20260612_134308/ga_output/advisor_prompt_generation_001.txt
advisor_prompt_generation_002.txt /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_cloud_only_by_config_A6000_SET_A_cloud_only_by_config_cat3456_lpc1_pop5_gens5_qwen25_coder_14b_20260612_134308/ga_output/advisor_prompt_generation_002.txt
advisor_prompt_generation_003.txt /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_cloud_only_by_config_A6000_SET_A_cloud_only_by_config_cat3456_lpc1_pop5_gens5_qwen25_coder_14b_2026061

### generation 5 다음에 generation 6만 이어 실행하는 Cell

In [ ]:
# ============================================================
# Resume same cloud-only-by-config run by one generation
# Example: after gens=5, run generation 6 only.
# ============================================================

from pathlib import Path
import subprocess
import shlex
import json
import os

OUT_DIR = Path(cloud_only_by_config_out)

NEXT_GEN = 6  # 5 다음이면 6, 그 다음에는 7로 바꿔서 실행

cmd = [
    str(PYTHON),
    "-u",
    str(SCRIPT),

    "--profile", "version0_15",
    "--model-key", SMOKE_MODEL_KEY,
    "--target-detpass", "90",
    "--llm-mode", "worker",

    "--population", str(SMOKE_POPULATION),
    "--gens", str(NEXT_GEN),
    "--min-generations", str(NEXT_GEN),
    "--max-generations", str(NEXT_GEN),

    "--sample-size", str(SMOKE_SAMPLE_SIZE),
    "--validation-size", str(SMOKE_VALIDATION_SIZE),
    "--cheap-eval-limit", "2",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",

    "--selection-mode", "redesign",
    "--fitness-mode", "phase_aware",
    "--mutation-mode", "cloudless_decompiler",
    "--category-balance-mode", "guard",
    "--token-penalty-mode", "hybrid",
    "--stop-controller-mode", "active",
    "--plateau-window", "1",
    "--disruptive-max-attempts", "1",
    "--reasoning-mutation-mode", "auto",
    "--intent-hint-mode", "auto",

    "--progress", "verbose",
    "--timeout-sec", "2400",
    "--retries", "0",
    "--limit-per-category", str(SMOKE_LIMIT_PER_CATEGORY),
    "--output-root", str(OUT_DIR),

    "--feedback-guided-mutation",
    "--enable-compression-mutation",
    "--enable-prompt-decompiler",
    "--enable-rendered-prompt-dedupe",
    "--enable-pareto-archive",
    "--enable-group-specialist-archives",
    "--full-run",
    "--resume",

    # cloud-only-by-config:
    # local compression lanes OFF
    "--compression-child-quota", "0",
    "--compression-child-ratio", "0.0",
    "--micro-compression-child-quota", "0",
    "--micro-compression-child-ratio", "0.0",
    "--block-compression-child-quota", "0",
    "--block-compression-child-ratio", "0.0",
    "--multi-block-compression-child-quota", "0",
    "--multi-block-compression-child-ratio", "0.0",
    "--global-budget-compression-child-quota", "0",

    # advisor compression ON
    "--llm-mutation-advisor",
    "--advisor-model-key", ADVISOR_MODEL_KEY,
    "--advisor-trigger-mode", "always",
    "--advisor-min-population-for-child", "4",
    "--advisor-force-child-quota",
    "--advisor-compression-child-quota", "2",
    "--advisor-prefer-compression-after-detpass", "90",

    "--compression-detpass-threshold", "90",
    "--aggressive-compression-after-target",
    "--compression-token-reduction-target", "0.15",
    "--compression-token-plateau-delta", "1.0",
    "--allow-aggressive-compression",
    "--enable-block-token-breakdown",
    "--enable-multi-block-compression",
    "--min-compression-token-delta", "50",
]

for c in SMOKE_CATEGORIES:
    cmd += ["--category", str(c)]

env = os.environ.copy()
env["JOI_V15_LOCAL_MODEL_BASE_DIR"] = str(LOCAL_MODEL_BASE)
env["JOI_V15_LOCAL_FILES_ONLY"] = "true"
env["TRANSFORMERS_OFFLINE"] = env.get("TRANSFORMERS_OFFLINE", "1")
env["HF_HUB_OFFLINE"] = env.get("HF_HUB_OFFLINE", "1")
env["TOKENIZERS_PARALLELISM"] = "false"

print("=" * 120)
print("[RESUME ONE GENERATION]")
print("=" * 120)
print("OUT_DIR:", OUT_DIR)
print("NEXT_GEN:", NEXT_GEN)
print("COMMAND:")
print(" ".join(shlex.quote(x) for x in cmd))

p = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
    timeout=7200,
)

print(p.stdout)
print("returncode:", p.returncode)

if p.returncode != 0:
    raise RuntimeError(f"resume generation {NEXT_GEN} failed")

### generation 5와 6 advisor prompt 비교

In [ ]:
# ============================================================
# Compare advisor prompt generation 005 vs 006
# ============================================================

from pathlib import Path
import difflib

OUT_DIR = Path(cloud_only_by_config_out)

before = OUT_DIR / "advisor_prompt_generation_005.txt"
after = OUT_DIR / "advisor_prompt_generation_006.txt"

print("before exists:", before.exists(), before)
print("after exists :", after.exists(), after)

if before.exists() and after.exists():
    before_lines = before.read_text(encoding="utf-8", errors="replace").splitlines()
    after_lines = after.read_text(encoding="utf-8", errors="replace").splitlines()

    print("before chars:", sum(len(x) for x in before_lines))
    print("after chars :", sum(len(x) for x in after_lines))
    print("delta chars :", sum(len(x) for x in after_lines) - sum(len(x) for x in before_lines))

    diff = difflib.unified_diff(
        before_lines,
        after_lines,
        fromfile=before.name,
        tofile=after.name,
        lineterm="",
        n=5,
    )

    diff_text = "\n".join(list(diff)[:500])
    print(diff_text)
else:
    print("[WARN] Need both generation 005 and 006 advisor prompt files.")

### generation 5 대비 6의 JOI prompt/block before-after 확인

#### A. block/genome diff 확인

In [ ]:
# ============================================================
# Inspect GA block diffs around generation 6
# ============================================================

from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path(cloud_only_by_config_out)
diff_path = OUT_DIR / "ga_block_diffs.jsonl"

rows = []
if diff_path.exists():
    for line in diff_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
            rows.append(obj)
        except Exception:
            pass

df_diffs = pd.DataFrame(rows)
display(df_diffs.tail(50))

if "generation" in df_diffs.columns:
    display(df_diffs[df_diffs["generation"].isin([5, 6, 7])])
else:
    print("[WARN] generation column not found in ga_block_diffs.jsonl")

#### B. token / DETPass 변화 확인

In [ ]:
# ============================================================
# Compare progress around generation 5, 6, 7
# ============================================================

from pathlib import Path
import pandas as pd

OUT_DIR = Path(cloud_only_by_config_out)
progress = pd.read_csv(OUT_DIR / "ga_generation_progress.csv")

cols = [
    "generation",
    "validation_det_pass_rate",
    "validation_avg_det_score",
    "best_so_far_DETPass",
    "avg_prompt_tokens",
    "compression_ready",
    "compression_phase",
    "advisor_triggered",
]

display(progress[[c for c in cols if c in progress.columns]].tail(10))

#### C. 실제 JOI code 생성 prompt log 비교

In [ ]:
# ============================================================
# Find actual JOI generation prompt logs from candidate CSVs
# ============================================================

from pathlib import Path
import pandas as pd
import ast

OUT_DIR = Path(cloud_only_by_config_out)
cand_dir = OUT_DIR / "candidates"

prompt_log_rows = []

if cand_dir.exists():
    for csv_path in sorted(cand_dir.glob("*.csv")):
        try:
            df = pd.read_csv(csv_path)
        except Exception:
            continue

        if "prompt_log_paths" not in df.columns:
            continue

        for _, row in df.iterrows():
            raw = row.get("prompt_log_paths")
            if pd.isna(raw):
                continue

            try:
                parsed = ast.literal_eval(raw)
                paths = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                paths = [raw]

            for p in paths:
                p = str(p).strip()
                if not p:
                    continue
                pp = Path(p)
                prompt_log_rows.append({
                    "candidate_csv": csv_path.name,
                    "prompt_log_path": str(pp),
                    "exists": pp.exists(),
                    "size": pp.stat().st_size if pp.exists() else None,
                })

prompt_log_df = pd.DataFrame(prompt_log_rows)
display(prompt_log_df.tail(50))

In [ ]:
# ============================================================
# Compare actual JOI generation prompts before/after
# Pick two prompt_log_path rows manually if needed.
# ============================================================

import difflib
from pathlib import Path

valid_logs = prompt_log_df[prompt_log_df["exists"] == True]["prompt_log_path"].tolist()

if len(valid_logs) >= 2:
    before_path = Path(valid_logs[-2])
    after_path = Path(valid_logs[-1])

    before_lines = before_path.read_text(encoding="utf-8", errors="replace").splitlines()
    after_lines = after_path.read_text(encoding="utf-8", errors="replace").splitlines()

    print("BEFORE:", before_path)
    print("AFTER :", after_path)
    print("before chars:", sum(len(x) for x in before_lines))
    print("after chars :", sum(len(x) for x in after_lines))
    print("delta chars :", sum(len(x) for x in after_lines) - sum(len(x) for x in before_lines))

    diff = difflib.unified_diff(
        before_lines,
        after_lines,
        fromfile=str(before_path),
        tofile=str(after_path),
        lineterm="",
        n=5,
    )

    print("\n".join(list(diff)[:500]))
else:
    print("[WARN] Need at least two actual prompt log files.")

In [9]:
# ============================================================
# Cell 9. Dataset evaluation feedback inspection
# ============================================================

def select_advisor_out_dir():
    if "cloud_only_by_config_out" in globals():
        return Path(cloud_only_by_config_out)
    if "cloud_only_out" in globals():
        return Path(cloud_only_out)
    raise RuntimeError("No cloud_only_by_config_out found. Run Cell 8 cloud-only-by-config advisor smoke first.")

OUT_DIR = select_advisor_out_dir()
print("OUT_DIR:", OUT_DIR)

print("\n[strict advisor_prompt_case_sections_generation_*.json, optional]")
rows = []
for p in sorted(OUT_DIR.glob("advisor_prompt_case_sections_generation_*.json")):
    obj = read_json(p) or {}
    rows.append({"file": p.name, "keys": list(obj.keys()), "has_DATASET_EVALUATION_FEEDBACK": "DATASET_EVALUATION_FEEDBACK" in obj})
display(pd.DataFrame(rows) if rows else pd.DataFrame([{"message":"No strict prompt case section files. This is expected in cloud-only-by-config mode."}]))

print("\n[advisor_feedback_batches.jsonl]")
batches = read_jsonl(OUT_DIR/"advisor_feedback_batches.jsonl")
display(pd.DataFrame(batches).head(20) if batches else pd.DataFrame([{"message":"No advisor feedback batches"}]))

print("\n[Prompt excerpt: DATASET_EVALUATION_FEEDBACK]")
for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt"))[:3]:
    txt = p.read_text(encoding="utf-8", errors="replace")
    idx = txt.find("DATASET_EVALUATION_FEEDBACK")
    print("\n" + "-"*120)
    print(p.name, "chars:", len(txt))
    if idx >= 0:
        print(txt[max(0, idx-400):idx+2600])
    else:
        print("[NOT FOUND]")


OUT_DIR: /home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/smoke_cloud_only_by_config_A6000_SET_A_cloud_only_by_config_cat3456_lpc1_pop5_gens5_qwen25_coder_14b_20260612_134308/ga_output

[strict advisor_prompt_case_sections_generation_*.json, optional]


,message
0,No strict prompt case section files. This is e...



[advisor_feedback_batches.jsonl]


,advisor_batch_id,generation,model_key,advisor_model_key,categories,limit_per_category,sample_size,validation_size,generation_phase,plateau_type,...,overall,category_diagnostics,group_diagnostics,genome_diagnostics,representative_failures,current_prompt_artifact,prompt_token_breakdown,block_token_breakdown,cloudless_feedback_summary,advisor_request
0,advisor_batch_g001_0cc56be3,1,qwen25_coder_14b,gpt41_mini,"[3, 4, 5, 6]",1,4,4,ACCURACY_SEARCH,warming_up,...,"{'best_DETPass': 75.0, 'best_AvgDET': 73.4821,...","[{'generation': 1, 'model_key': 'qwen25_coder_...","{'basic': {'categories': [], 'row_count': 0, '...","{'top_genomes': [{'rank': 1, 'genome_id': 'gen...","[{'row_id': 61, 'category': 3, 'group': 'tempo...",{'genome_id': 'gen-af17dfa2-8346-1468-fe4c-138...,"{'generation': 1, 'model_key': 'qwen25_coder_1...","[{'generation': 1, 'model_key': 'qwen25_coder_...","{'structured_feedback_count': 10, 'applied_clo...",{'goal': 'Generate structured mutation proposa...
1,advisor_batch_g002_3360bc3b,2,qwen25_coder_14b,gpt41_mini,"[3, 4, 5, 6]",1,4,4,ACCURACY_SEARCH,warming_up,...,"{'best_DETPass': 75.0, 'best_AvgDET': 74.4288,...","[{'generation': 2, 'model_key': 'qwen25_coder_...","{'basic': {'categories': [], 'row_count': 0, '...","{'top_genomes': [{'rank': 1, 'genome_id': 'gen...","[{'row_id': 61, 'category': 3, 'group': 'tempo...",{'genome_id': 'gen-691877a4-7185-0d8f-e71f-4e6...,"{'generation': 2, 'model_key': 'qwen25_coder_1...","[{'generation': 2, 'model_key': 'qwen25_coder_...","{'structured_feedback_count': 10, 'applied_clo...",{'goal': 'Generate structured mutation proposa...
2,advisor_batch_g003_94b9e5ca,3,qwen25_coder_14b,gpt41_mini,"[3, 4, 5, 6]",1,4,4,ACCURACY_SEARCH,warming_up,...,"{'best_DETPass': 75.0, 'best_AvgDET': 74.4288,...","[{'generation': 3, 'model_key': 'qwen25_coder_...","{'basic': {'categories': [], 'row_count': 0, '...","{'top_genomes': [{'rank': 1, 'genome_id': 'gen...","[{'row_id': 151, 'category': 6, 'group': 'comp...",{'genome_id': 'gen-691877a4-7185-0d8f-e71f-4e6...,"{'generation': 3, 'model_key': 'qwen25_coder_1...","[{'generation': 3, 'model_key': 'qwen25_coder_...","{'structured_feedback_count': 9, 'applied_clou...",{'goal': 'Generate structured mutation proposa...
3,advisor_batch_g004_bb736d69,4,qwen25_coder_14b,gpt41_mini,"[3, 4, 5, 6]",1,4,4,ACCURACY_SEARCH,warming_up,...,"{'best_DETPass': 75.0, 'best_AvgDET': 74.4288,...","[{'generation': 4, 'model_key': 'qwen25_coder_...","{'basic': {'categories': [], 'row_count': 0, '...","{'top_genomes': [{'rank': 1, 'genome_id': 'gen...","[{'row_id': 151, 'category': 6, 'group': 'comp...",{'genome_id': 'gen-65f78f51-2caf-55da-9da0-1b1...,"{'generation': 4, 'model_key': 'qwen25_coder_1...","[{'generation': 4, 'model_key': 'qwen25_coder_...","{'structured_feedback_count': 10, 'applied_clo...",{'goal': 'Generate structured mutation proposa...
4,advisor_batch_g005_c3d0935d,5,qwen25_coder_14b,gpt41_mini,"[3, 4, 5, 6]",1,4,4,FINAL_SELECTION,max_generation_reached,...,"{'best_DETPass': 75.0, 'best_AvgDET': 74.4288,...","[{'generation': 5, 'model_key': 'qwen25_coder_...","{'basic': {'categories': [], 'row_count': 0, '...","{'top_genomes': [{'rank': 1, 'genome_id': 'gen...","[{'row_id': 151, 'category': 6, 'group': 'comp...",{'genome_id': 'gen-765a9f65-11c3-c8e3-cfea-0b7...,"{'generation': 5, 'model_key': 'qwen25_coder_1...","[{'generation': 5, 'model_key': 'qwen25_coder_...","{'structured_feedback_count': 9, 'applied_clou...",{'goal': 'Generate structured mutation proposa...



[Prompt excerpt: DATASET_EVALUATION_FEEDBACK]

------------------------------------------------------------------------------------------------------------------------
advisor_prompt_generation_001.txt chars: 39036
[NOT FOUND]

------------------------------------------------------------------------------------------------------------------------
advisor_prompt_generation_002.txt chars: 40233
[NOT FOUND]

------------------------------------------------------------------------------------------------------------------------
advisor_prompt_generation_003.txt chars: 32881
[NOT FOUND]


In [ ]:
# ============================================================
# Cell 10. Exact advisor prompt inspection
# ============================================================

OUT_DIR = select_advisor_out_dir()
sections = [
    "ADVISOR_ROLE", "CURRENT_STATE", "DATASET_EVALUATION_FEEDBACK",
    "PROMPT_TOKEN_BREAKDOWN", "BLOCK_TOKEN_BREAKDOWN", "PROTECTED_BLOCKS",
    "COMPRESSION_ALLOWED_BLOCKS", "CASE_A_MICRO_COMPRESSION",
    "CASE_B_THRESHOLD_BLOCK_COMPRESSION", "CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION",
    "STRICT_JSON_RESPONSE_SCHEMA",
]

def extract_section_text(text, section, max_chars=3000):
    idx = text.find(section)
    if idx < 0:
        return ""
    end = min(len(text), idx + max_chars)
    nexts = [text.find(s, idx + len(section)) for s in sections if s != section and text.find(s, idx + len(section)) > idx]
    if nexts:
        end = min(end, min(nexts))
    return text[idx:end]

prompt_rows = []
for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt")):
    txt = p.read_text(encoding="utf-8", errors="replace")
    row = {"file": p.name, "chars": len(txt), "approx_tokens": round(len(txt)/4)}
    for s in sections:
        row[f"has_{s}"] = s in txt
    row["has_advisor_model_key"] = "advisor_model_key" in txt or "gpt41_mini" in txt
    prompt_rows.append(row)

prompt_summary_df = pd.DataFrame(prompt_rows)
display(prompt_summary_df)

for p in sorted(OUT_DIR.glob("advisor_prompt_generation_*.txt"))[:2]:
    txt = p.read_text(encoding="utf-8", errors="replace")
    print("\n" + "="*120)
    print("PROMPT:", p.name, "chars:", len(txt), "approx_tokens:", round(len(txt)/4))
    for s in sections:
        sec = extract_section_text(txt, s, 2000)
        if sec:
            print("\n" + "-"*100)
            print(sec[:2000])


In [ ]:
# ============================================================
# Cell 11. Advisor raw response inspection
# ============================================================

OUT_DIR = select_advisor_out_dir()
proposal_keys = [
    "proposals", "mutation_proposals", "micro_compression_proposals",
    "block_compression_proposals", "multi_block_compression_proposals",
    "global_budget_compression_proposals",
]
resp_rows, raw_prop_rows = [], []

for p in sorted(OUT_DIR.glob("advisor_response_generation_*.json")):
    obj = read_json(p) or {}
    parsed = obj.get("parsed", {})
    raw = str(obj.get("raw_content", ""))
    m = re.search(r"generation_(\d+)", p.name)
    gen = int(m.group(1)) if m else None
    row = {
        "generation": gen, "file": p.name, "size_bytes": p.stat().st_size,
        "raw_chars": len(raw), "parsed_type": type(parsed).__name__,
        "accepted_count": len(obj.get("accepted_proposals", []) or []),
        "rejected_count": len(obj.get("rejected_proposals", []) or []),
    }
    if isinstance(parsed, dict):
        row.update({
            "advisor_status": parsed.get("advisor_status"),
            "advisor_model_key": parsed.get("advisor_model_key") or obj.get("advisor_model_key"),
            "dataset_feedback_seen": parsed.get("dataset_feedback_seen"),
            "prompt_token_breakdown_seen": parsed.get("prompt_token_breakdown_seen"),
            "block_token_breakdown_seen": parsed.get("block_token_breakdown_seen"),
        })
        for k in proposal_keys:
            v = parsed.get(k)
            row[k+"_count"] = len(v) if isinstance(v, list) else 0
            if isinstance(v, list):
                for item in v:
                    if isinstance(item, dict):
                        rr = dict(item)
                        rr["generation"] = gen
                        rr["schema_source"] = k
                        rr["_response_file"] = p.name
                        rr["operator"] = rr.get("operator", rr.get("mutation_type", rr.get("mutation", "")))
                        rr["case_label_inferred"] = classify_case_abc(rr)
                        raw_prop_rows.append(rr)
    resp_rows.append(row)

advisor_response_df = pd.DataFrame(resp_rows)
display(advisor_response_df)

raw_advisor_proposals_df = pd.DataFrame(raw_prop_rows)
print("raw advisor proposal rows:", len(raw_advisor_proposals_df))
if len(raw_advisor_proposals_df):
    cols = [c for c in ["generation","_response_file","schema_source","case_label","case_label_inferred","compression_level","operator","selected_block_id","selected_block_ids","expected_token_delta","dataset_feedback_reason","reason"] if c in raw_advisor_proposals_df.columns]
    display(raw_advisor_proposals_df[cols])

print("\n[Repair traces]")
repair_files = sorted(OUT_DIR.glob("advisor_proposal_repair_trace_generation_*.json"))
print("count:", len(repair_files))
for p in repair_files[:5]:
    print(p.name, p.stat().st_size, "bytes", list((read_json(p) or {}).keys()))

In [ ]:
# ============================================================
# Cell 12. Unified proposal table
# ============================================================

OUT_DIR = select_advisor_out_dir()
proposal_df = load_all_proposals(OUT_DIR)

if len(proposal_df):
    cols = [c for c in [
        "_file","generation","source","schema_source","case_label","case_label_inferred",
        "operator","compression_level","selected_block_id","selected_block_ids",
        "expected_token_delta","measured_prompt_token_delta","accepted","applied",
        "rejection_reason","fallback_reason","dataset_feedback_reason"
    ] if c in proposal_df.columns]
    display(proposal_df[cols])
    group_cols = [c for c in ["source","case_label_inferred","operator"] if c in proposal_df.columns]
    if group_cols:
        display(proposal_df.groupby(group_cols, dropna=False).size().reset_index(name="count"))
else:
    print("[WARN] no proposals found")

In [ ]:
# ============================================================
# Cell 13. Before/after block and prompt comparison
# ============================================================

OUT_DIR = select_advisor_out_dir()

before_after_path = OUT_DIR / "advisor_cloud_only_before_after.csv"
if before_after_path.exists():
    before_after_df = pd.read_csv(before_after_path)
    print("[advisor_cloud_only_before_after.csv]")
    display(before_after_df)
else:
    print("[WARN] advisor_cloud_only_before_after.csv not found")
    before_after_df = pd.DataFrame()

def flatten_block_diff(obj):
    rows = []
    base = {k: obj.get(k) for k in ["generation","genome_id","parent","child","proposal_id"]}
    lists = []
    for k in ["changed","changes","diffs","block_diffs","mutations"]:
        if isinstance(obj.get(k), list):
            lists.extend(obj[k])
    if not lists and any(k in obj for k in ["block","block_id","old","new","mutation"]):
        lists = [obj]
    for d in lists:
        if not isinstance(d, dict):
            continue
        row = dict(base)
        row.update({
            "block": d.get("block", d.get("block_id", d.get("target_block_id"))),
            "mutation": d.get("mutation", d.get("operator", d.get("mutation_type"))),
            "old": d.get("old"),
            "new": d.get("new"),
            "source": d.get("source"),
            "compression_level": d.get("compression_level"),
        })
        row["case_label_inferred"] = classify_case_abc(row)
        rows.append(row)
    return rows

diff_rows = []
for fn in ["ga_block_diffs.jsonl", "advisor_accepted_block_diffs.jsonl"]:
    for obj in read_jsonl(OUT_DIR/fn):
        diff_rows.extend(flatten_block_diff(obj))
block_diff_df = pd.DataFrame(diff_rows)

print("\n[BLOCK DIFFS]")
display(block_diff_df if len(block_diff_df) else pd.DataFrame([{"message":"No block diffs parsed"}]))

print("\n[PROMPT BEFORE/AFTER FILES]")
for p in sorted(OUT_DIR.glob("advisor_prompt_before_after_generation_*.json"))[:10]:
    obj = read_json(p) or {}
    print(p.name, {k: obj.get(k) for k in ["parent_prompt_tokens","child_prompt_tokens","token_delta","changed_blocks","removed_blocks","changed_few_shot_count","changed_max_tokens"] if k in obj})

In [ ]:
# ============================================================
# Cell 14. Prompt token and DETPass trend
# ============================================================

OUT_DIR = select_advisor_out_dir()
progress = load_progress(OUT_DIR)
trans = load_transitions(OUT_DIR)

if len(progress):
    cols = [c for c in ["generation","validation_det_pass_rate","validation_avg_det_score","best_so_far_DETPass","avg_prompt_tokens","compression_ready","compression_phase"] if c in progress.columns]
    display(progress[cols])
    try:
        import matplotlib.pyplot as plt
        if "generation" in progress.columns and "avg_prompt_tokens" in progress.columns:
            plt.figure()
            plt.plot(progress["generation"], pd.to_numeric(progress["avg_prompt_tokens"], errors="coerce"), marker="o")
            plt.xlabel("Generation")
            plt.ylabel("Average prompt tokens")
            plt.title("Prompt token trend")
            plt.grid(True, alpha=0.3)
            plt.show()
        det_col = "best_so_far_DETPass" if "best_so_far_DETPass" in progress.columns else "validation_det_pass_rate"
        if "generation" in progress.columns and det_col in progress.columns:
            plt.figure()
            plt.plot(progress["generation"], pd.to_numeric(progress[det_col], errors="coerce"), marker="o")
            plt.xlabel("Generation")
            plt.ylabel(det_col)
            plt.title("DETPass trend")
            plt.grid(True, alpha=0.3)
            plt.show()
    except Exception as e:
        print("[WARN] plot failed:", repr(e))
else:
    print("[WARN] no progress")

In [ ]:
# ============================================================
# Cell 15. Case A/B/C summary
# ============================================================

OUT_DIR = select_advisor_out_dir()
case_path = OUT_DIR / "advisor_case_ABC_summary.csv"
if case_path.exists():
    case_summary_df = pd.read_csv(case_path)
else:
    prop = load_all_proposals(OUT_DIR)
    rows = []
    if len(prop):
        for case, g in prop.groupby("case_label_inferred", dropna=False):
            rows.append({
                "case_label": case,
                "proposals_generated": len(g),
                "proposals_accepted": int((g.get("accepted", pd.Series(False, index=g.index)) == True).sum()) if "accepted" in g.columns else np.nan,
                "proposals_rejected": int((g.get("accepted", pd.Series(False, index=g.index)) == False).sum()) if "accepted" in g.columns else np.nan,
                "mean_expected_token_delta": pd.to_numeric(g.get("expected_token_delta"), errors="coerce").mean() if "expected_token_delta" in g.columns else np.nan,
                "mean_measured_token_delta": pd.to_numeric(g.get("measured_prompt_token_delta"), errors="coerce").mean() if "measured_prompt_token_delta" in g.columns else np.nan,
                "most_common_operator": g["operator"].mode().iloc[0] if "operator" in g.columns and len(g["operator"].dropna()) else None,
            })
    case_summary_df = pd.DataFrame(rows)
display(case_summary_df if len(case_summary_df) else pd.DataFrame([{"message":"No Case A/B/C summary"}]))

In [ ]:
# ============================================================
# Cell 16. Three-way comparison
# ============================================================

runs = []
if "cloudless_out" in globals(): runs.append(("cloudless", cloudless_out))
if "hybrid_out" in globals(): runs.append(("hybrid_advisor", hybrid_out))
if "cloud_only_out" in globals(): runs.append(("cloud_only_advisor", cloud_only_out))

if runs:
    three_way_df = pd.DataFrame([summarize_run(out, label=label, mode=label) for label, out in runs])
    display(three_way_df)
    out_csv = SUMMARY_DIR / f"cloudless_vs_hybrid_vs_cloudonly_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    three_way_df.to_csv(out_csv, index=False)
    print("saved:", out_csv)
else:
    print("[WARN] no run variables available")

In [ ]:
# ============================================================
# Cell 17. Final gates
# Cloud-only-by-config gate: fallback must be zero, strict artifacts are optional.
# ============================================================

def gate_common(out_dir, threshold=90):
    errors, warnings = [], []
    out_dir = Path(out_dir)
    p = load_progress(out_dir)
    t = load_transitions(out_dir)

    for fn in ["ga_summary.json", "ga_generation_progress.csv", "population_transitions.csv"]:
        if not (out_dir / fn).exists():
            errors.append(f"missing {fn}")

    if len(p):
        if "avg_prompt_tokens" not in p.columns or pd.to_numeric(p["avg_prompt_tokens"], errors="coerce").fillna(0).max() <= 0:
            errors.append("avg_prompt_tokens missing or non-positive")

        dets = []
        for c in ["best_so_far_DETPass", "validation_det_pass_rate", "det_pass_rate"]:
            if c in p.columns:
                dets.append(float(pd.to_numeric(p[c], errors="coerce").fillna(0).max()))
        if (max(dets) if dets else 0) < threshold:
            errors.append(f"DETPass below threshold {threshold}")
    else:
        errors.append("progress empty")

    return errors, warnings

def gate_cloud_only_by_config(out_dir):
    errors, warnings = gate_common(out_dir)
    out_dir = Path(out_dir)
    prop = load_all_proposals(out_dir)
    trans = load_transitions(out_dir)

    # The defining property of cloud-only-by-config:
    # local compression quotas are zero and fallback must not appear in artifacts.
    fallback_rows = 0
    if len(prop) and "source" in prop.columns:
        fallback_rows = int((prop["source"].astype(str) == "compression_fallback").sum())
        if fallback_rows:
            errors.append(f'fallback rows exist in cloud-only-by-config run: {fallback_rows}')

    if len(trans) and "new_by_compression_fallback" in trans.columns:
        fb_sum = float(pd.to_numeric(trans["new_by_compression_fallback"], errors="coerce").fillna(0).sum())
        if fb_sum != 0:
            errors.append(f"new_by_compression_fallback_sum={fb_sum}")
    else:
        warnings.append("new_by_compression_fallback column not found; fallback row check is used instead.")

    advisor_rows = int((prop.get("_file", pd.Series(dtype=str)) == "advisor_mutation_proposals.jsonl").sum()) if len(prop) else 0
    if advisor_rows <= 0:
        errors.append("advisor proposal rows == 0")

    accepted = int((prop.get("accepted", pd.Series(dtype=bool)) == True).sum()) if len(prop) and "accepted" in prop.columns else 0
    applied = int((prop.get("applied", pd.Series(dtype=bool)) == True).sum()) if len(prop) and "applied" in prop.columns else 0
    rejected = int((prop.get("accepted", pd.Series(dtype=object)) == False).sum()) if len(prop) and "accepted" in prop.columns else 0

    if accepted + applied <= 0:
        warnings.append(
            "advisor accepted/applied count is 0. The run can still prove cloud-only-by-config isolation, "
            "but advisor proposal quality may be weak."
        )

    for optional_artifact in ["advisor_cloud_only_before_after.csv", "advisor_case_ABC_summary.csv"]:
        if not (out_dir / optional_artifact).exists():
            warnings.append(f"{optional_artifact} not found. This is expected without strict cloud-only implementation.")

    print("fallback_rows:", fallback_rows)
    print("advisor_rows:", advisor_rows)
    print("advisor_accepted:", accepted)
    print("advisor_applied:", applied)
    print("advisor_rejected:", rejected)

    return errors, warnings

targets = []
if "cloudless_out" in globals():
    targets.append(("cloudless", cloudless_out, gate_common))
if "hybrid_out" in globals():
    targets.append(("hybrid", hybrid_out, gate_common))
if "cloud_only_by_config_out" in globals():
    targets.append(("cloud_only_by_config", cloud_only_by_config_out, gate_cloud_only_by_config))
elif "cloud_only_out" in globals():
    targets.append(("cloud_only_by_config", cloud_only_out, gate_cloud_only_by_config))

if not targets:
    print("[WARN] no run variables found")
else:
    for name, out, fn in targets:
        print("\n" + "=" * 120)
        print(f"[FINAL GATE: {name}]")
        errors, warnings = fn(out)

        if warnings:
            print("[WARNINGS]")
            for w in warnings:
                print("-", w)

        if errors:
            print("[DO NOT PROMOTE]")
            for e in errors:
                print("-", e)
        else:
            print("[OK]", name, "gate passed")


In [ ]:
# ============================================================
# Cell 18. Optional full run command cells
# Disabled by default.
# ============================================================

FULL_COMMON = dict(
    categories=tuple(range(1, 9)),
    limit_per_category=3,
    sample_size=24,
    validation_size=24,
    population=5,
    gens=10,
    target_detpass=90,
    full_run=True,
    progress="verbose",
    timeout_sec=3600,
    retries=0,
)

def run_full_cloud_only_by_config(label, model_key):
    return run_ga_all_categories(
        model_key=model_key,
        base_prefix=f"ga_{SERVER_PRESET}_cloud_only_by_config",
        use_advisor=True,
        advisor_model_key="gpt41_mini",
        advisor_trigger_mode="always",
        advisor_min_population_for_child=4,
        advisor_force_child_quota=True,
        cloud_only_advisor_compression=False,
        disable_compression_fallback=False,
        compression_child_quota=0,
        compression_child_ratio=0.0,
        micro_compression_child_quota=0,
        micro_compression_child_ratio=0.0,
        block_compression_child_quota=0,
        block_compression_child_ratio=0.0,
        multi_block_compression_child_quota=0,
        multi_block_compression_child_ratio=0.0,
        global_budget_compression_child_quota=0,
        advisor_compression_child_quota=2,
        advisor_proposal_k=6,
        advisor_min_usable_compression_proposals=2,
        advisor_repair_invalid_proposals=True,
        advisor_repair_max_attempts=2,
        advisor_require_block_proposal_after_detpass=True,
        advisor_require_token_delta=True,
        advisor_disallow_genome_only_after_detpass=True,
        advisor_cloud_only_strict_schema=True,
        advisor_before_after_report=True,
        advisor_include_dataset_feedback=True,
        advisor_feedback_topk=5,
        advisor_feedback_max_failures_per_family=3,
        advisor_include_prompt_before_after_context=True,
        advisor_include_case_abc_sections=True,
        advisor_write_debug_prompt_sections=True,
        **FULL_COMMON,
    )

if RUN_FULL_CLOUD_ONLY:
    full_cloud_only_runs = {}
    for label, model_key in MODEL_LIST_3:
        full_cloud_only_runs[label] = run_full_cloud_only(label, model_key)
else:
    print("[SKIP] RUN_FULL_CLOUD_ONLY=False")

if RUN_FULL_THREE_WAY:
    full_three_way_runs = {}
    for label, model_key in MODEL_LIST_3:
        full_three_way_runs[f"{label}_cloudless"] = run_ga_all_categories(model_key=model_key, base_prefix=f"ga_{SERVER_PRESET}_cloudless", use_advisor=False, **FULL_COMMON)
        full_three_way_runs[f"{label}_hybrid"] = run_ga_all_categories(model_key=model_key, base_prefix=f"ga_{SERVER_PRESET}_hybrid", use_advisor=True, advisor_model_key="gpt41_mini", **FULL_COMMON)
        full_three_way_runs[f"{label}_cloud_only"] = run_full_cloud_only(label, model_key)
else:
    print("[SKIP] RUN_FULL_THREE_WAY=False")


In [ ]:
# ============================================================
# Cell 19. Save paper artifacts
# ============================================================

def save_prompt_sections_md(out_dir, output_path):
    out_dir = Path(out_dir)
    lines = []
    for p in sorted(out_dir.glob("advisor_prompt_generation_*.txt")):
        txt = p.read_text(encoding="utf-8", errors="replace")
        lines.append(f"# {p.name}\n")
        for s in ["CURRENT_STATE","DATASET_EVALUATION_FEEDBACK","CASE_A_MICRO_COMPRESSION","CASE_B_THRESHOLD_BLOCK_COMPRESSION","CASE_C_AGGRESSIVE_MULTI_GLOBAL_COMPRESSION","STRICT_JSON_RESPONSE_SCHEMA"]:
            sec = extract_section_text(txt, s, 3500) if "extract_section_text" in globals() else ""
            if sec:
                lines.append(f"## {s}\n\n```text\n{sec}\n```\n")
    Path(output_path).write_text("\n".join(lines), encoding="utf-8")

saved = []
runs = []
if "cloudless_out" in globals(): runs.append(("cloudless", cloudless_out))
if "hybrid_out" in globals(): runs.append(("hybrid_advisor", hybrid_out))
if "cloud_only_out" in globals(): runs.append(("cloud_only_advisor", cloud_only_out))

if runs:
    main_df = pd.DataFrame([summarize_run(out, label=label, mode=label) for label, out in runs])
    display(main_df)
    p = TABLE_DIR / f"cloudless_vs_hybrid_vs_cloudonly_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    main_df.to_csv(p, index=False); saved.append(p)

if "cloud_only_out" in globals():
    out = Path(cloud_only_out)
    for src, prefix in [("advisor_cloud_only_before_after.csv","cloud_only_before_after"),("advisor_case_ABC_summary.csv","cloud_only_case_ABC_summary")]:
        sp = out/src
        if sp.exists():
            dp = TABLE_DIR / f"{prefix}_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            pd.read_csv(sp).to_csv(dp, index=False); saved.append(dp)
    mdp = TABLE_DIR / f"advisor_prompt_sections_{SERVER_PRESET}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
    save_prompt_sections_md(out, mdp); saved.append(mdp)

print("[SAVED]")
for p in saved:
    print(p)

In [ ]:
# ============================================================
# Cell 20. Interpretation template
# ============================================================

def build_interpretation(out_dir, mode_label):
    row = summarize_run(out_dir, label=mode_label, mode=mode_label)
    prop = load_all_proposals(out_dir)
    fallback_rows = row.get("fallback_rows", 0)
    case_counts = {}
    if len(prop):
        case_counts = prop.groupby("case_label_inferred").size().to_dict()

    kr = (
        f"[한국어 해석 템플릿]\n\n"
        f"본 실행은 {mode_label} 모드에서 수행되었다.\n"
        f"dataset evaluation 결과는 기존 advisor feedback batch 및 advisor prompt에 전달된다. strict cloud-only patch가 없으면 DATASET_EVALUATION_FEEDBACK이라는 명시 섹션명은 없을 수 있다.\n"
        f"이 섹션에는 DETPass, validation pass rate, token trend, failure family, target block mapping, "
        f"prompt_token_breakdown, block_token_breakdown 정보가 포함된다.\n\n"
        f"실행 결과:\n"
        f"- best_so_far_DETPass: {row.get('best_so_far_DETPass', row.get('validation_det_pass_rate'))}\n"
        f"- first prompt tokens: {row.get('first_tokens')}\n"
        f"- minimum prompt tokens: {row.get('min_tokens')}\n"
        f"- best token reduction ratio: {row.get('token_reduction_ratio_min')}\n"
        f"- advisor accepted rows: {row.get('advisor_accepted_rows')}\n"
        f"- advisor rejected rows: {row.get('advisor_rejected_rows')}\n"
        f"- fallback rows: {fallback_rows}\n"
        f"- Case A/B/C proposal counts: {case_counts}\n\n"
        f"해석 기준:\n"
        f"- fallback rows가 0이면 cloud-only advisor isolation 조건을 만족한다.\n"
        f"- fallback rows가 0보다 크면 advisor-only 기여로 주장하면 안 된다.\n"
        f"- token 감소는 measured_prompt_token_delta 또는 before/after artifact가 있을 때만 advisor 기여로 주장한다.\n"
        f"- accepted/applied proposal의 before-after block/genome/token diff가 있어야 cloud advisor 기반 prompt compression이라고 볼 수 있다.\n"
    )

    en = (
        f"[English interpretation template]\n\n"
        f"This run was executed in {mode_label} mode.\n"
        f"Dataset evaluation feedback is injected into the cloud advisor prompt through DATASET_EVALUATION_FEEDBACK.\n"
        f"The section includes DETPass, validation pass rate, token trend, failure families, target block mapping, "
        f"prompt_token_breakdown, and block_token_breakdown.\n\n"
        f"Observed results:\n"
        f"- best_so_far_DETPass: {row.get('best_so_far_DETPass', row.get('validation_det_pass_rate'))}\n"
        f"- first prompt tokens: {row.get('first_tokens')}\n"
        f"- minimum prompt tokens: {row.get('min_tokens')}\n"
        f"- best token reduction ratio: {row.get('token_reduction_ratio_min')}\n"
        f"- advisor accepted rows: {row.get('advisor_accepted_rows')}\n"
        f"- advisor rejected rows: {row.get('advisor_rejected_rows')}\n"
        f"- fallback rows: {fallback_rows}\n"
        f"- Case A/B/C proposal counts: {case_counts}\n\n"
        f"Interpretation criteria:\n"
        f"- fallback rows equal to zero means cloud-only advisor isolation is satisfied.\n"
        f"- nonzero fallback rows mean compression includes fallback contribution, not advisor-only contribution.\n"
        f"- token reduction should be attributed to advisor only when measured_prompt_token_delta or before/after artifacts exist.\n"
        f"- accepted/applied proposal-level before-after block/genome/token diff is required to claim cloud-advisor-driven compression.\n"
    )
    return kr, en

if "cloud_only_out" in globals():
    kr, en = build_interpretation(cloud_only_out, "cloud-only advisor")
    print(kr); print("\n" + "="*120 + "\n"); print(en)
elif "hybrid_out" in globals():
    kr, en = build_interpretation(hybrid_out, "hybrid advisor")
    print(kr); print("\n" + "="*120 + "\n"); print(en)
else:
    print("[WARN] Run cloud-only or hybrid smoke first.")
